# Querying `p(any column | any subset)` directly

Every estimator is a thin adapter over `TabularLanguageModel`, one fine-tune over permuted column
orders that learns $p(x_j \mid x_S)$ for any column $j$ and subset $S$. This notebook drives that
object directly:

- **Score** a fixed candidate set for one column with `predict_proba(known, target, candidates)`,
  what the classifier does.
- **Generate** a value for another column with `complete(known, targets, generation)`, what the
  imputer and regressor do.

The same fitted model answers both: `p(species | small petals)` and a sampled
`petal length | species=setosa` come from the same parameters.

In [ ]:
from sklearn.datasets import load_iris

from sklm import (
    GenerationConfig,
    JSONSerializer,
    JupyterCallback,
    MLXBackend,
    ModelConfig,
    TabularLanguageModel,
    TrainingConfig,
)

SEED = 42

## Data

The full Iris table, four measurements plus the species label, with no target singled out.

In [ ]:
iris = load_iris(as_frame=True)
frame = iris.data.round(1)
frame["species"] = iris.target_names[iris.target]
frame.head()

## Fit the core model

`fit` with no `target_cols` treats every column as a potential target. `JupyterCallback` renders a
live training dashboard inline.

In [ ]:
lm = TabularLanguageModel(
    backend=MLXBackend(),
    serializer=JSONSerializer(),
    model=ModelConfig(model="mlx-community/distilgpt2"),
    training=TrainingConfig(epochs=40, batch_size=16),
    callback=JupyterCallback(),
    random_state=SEED,
).fit(frame)

## Score a categorical column

Condition on the two petal measurements only and ask for the species distribution.

In [ ]:
known = {"petal length (cm)": 1.4, "petal width (cm)": 0.2}
proba = lm.predict_proba(known, "species", list(iris.target_names))
for c, p in zip(iris.target_names, proba, strict=True):
    print(f"p(species={c} | small petals) = {p:.3f}")

## Generate a numeric column

The other direction: condition on the species and generate a petal length. `complete` returns the
generated columns, or `None` if the output stayed malformed after the retries.

In [ ]:
out = lm.complete({"species": "setosa"}, ["petal length (cm)"], GenerationConfig())
sampled = out["petal length (cm)"] if out is not None else "(malformed)"
print(f"sampled petal length | species=setosa -> {sampled}")